# Gradient boosting — XGBoost + LightGBM cheatsheet


XGBoost and LightGBM share a near-identical API: instantiate, `.fit(X, y)`, `.predict()`/`.predict_proba()`. Differences are mostly performance (LightGBM is 2-5× faster and handles categoricals natively) and naming conventions for hyperparameters.

This cheatsheet covers both side-by-side, with notes on which hyperparameter does what and how to set up early stopping, quantile regression, and persistence.


---
## Setup

Run this once.


### Setup — run me first


In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
import lightgbm as lgb
from sklearn.datasets import make_classification, make_regression

rng = np.random.default_rng(0)
X_clf, y_clf = make_classification(n_samples=600, n_features=8, random_state=0)
X_reg, y_reg = make_regression(n_samples=600, n_features=4, noise=5, random_state=0)
X_clf = pd.DataFrame(X_clf, columns=[f'f{i}' for i in range(8)])
X_clf_tr, X_clf_va = X_clf[:400], X_clf[400:]
y_clf_tr, y_clf_va = y_clf[:400], y_clf[400:]

---
## 1. The fit/predict pattern (identical between XGB and LGBM)

Both libraries expose an sklearn-compatible API. Same code works on both with one import change.


### How do I fit a classifier?

Pick `XGBClassifier()` or `LGBMClassifier()`. Same fit signature, same predict_proba.


In [2]:
m_xgb = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=0).fit(X_clf_tr, y_clf_tr)
m_lgb = lgb.LGBMClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, verbosity=-1, random_state=0).fit(X_clf_tr, y_clf_tr)

print(f'XGB val acc:  {m_xgb.score(X_clf_va, y_clf_va):.3f}')
print(f'LGBM val acc: {m_lgb.score(X_clf_va, y_clf_va):.3f}')

XGB val acc:  0.950
LGBM val acc: 0.960


*Common mistake*: forgetting `verbosity=-1` on LightGBM and getting flooded with output. *Why both*: LightGBM is faster and handles categoricals natively; XGBoost is more mature, more deterministic, and integrates with more downstream tooling.


### What hyperparameters actually matter?

Three: `n_estimators`, `learning_rate`, `max_depth` (or `num_leaves` in LightGBM). Everything else is fine-tuning.


In [3]:
# Standard recipe that gets ~80% of the way to optimal on most tabular data.
recipe_xgb = dict(n_estimators=300, learning_rate=0.05, max_depth=5,
                   subsample=0.9, colsample_bytree=0.9, random_state=0)
recipe_lgb = dict(n_estimators=300, learning_rate=0.05, num_leaves=31, max_depth=5,
                   subsample=0.9, colsample_bytree=0.9, verbosity=-1, random_state=0)

m_xgb = xgb.XGBClassifier(**recipe_xgb).fit(X_clf_tr, y_clf_tr)
m_lgb = lgb.LGBMClassifier(**recipe_lgb).fit(X_clf_tr, y_clf_tr)
print(f'XGB val acc:  {m_xgb.score(X_clf_va, y_clf_va):.3f}')
print(f'LGBM val acc: {m_lgb.score(X_clf_va, y_clf_va):.3f}')

XGB val acc:  0.950
LGBM val acc: 0.955


*Trade-off*: `n_estimators` × `learning_rate` ≈ constant. Doubling `n_estimators` and halving `learning_rate` gives a similar fit at twice the cost. Start with `learning_rate=0.05-0.1` and let early stopping pick `n_estimators`.


### How do I use early stopping?

Pass `eval_set` to `.fit()` and use the per-library callback or kwarg.


In [4]:
# XGBoost (≥2.0)
m_xgb = xgb.XGBClassifier(n_estimators=500, learning_rate=0.05, early_stopping_rounds=20, eval_metric='logloss').fit(
    X_clf_tr, y_clf_tr, eval_set=[(X_clf_va, y_clf_va)], verbose=False,
)
print(f'XGB best iteration: {m_xgb.best_iteration}')

# LightGBM
m_lgb = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, verbosity=-1).fit(
    X_clf_tr, y_clf_tr, eval_set=[(X_clf_va, y_clf_va)],
    callbacks=[lgb.early_stopping(20), lgb.log_evaluation(0)],
)
print(f'LGBM best iteration: {m_lgb.best_iteration_}')

XGB best iteration: 96
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[77]	valid_0's binary_logloss: 0.137109
LGBM best iteration: 77


*Why early stopping*: prevents overfitting by stopping when val score stops improving. *Common mistake*: setting `n_estimators=10000` 'just in case' without early stopping — the model overfits dramatically.


---
## 2. Feature handling

Built-in handling of categoricals (LightGBM) and missing values (both).


### How do I tell LightGBM that a column is categorical?

Pass the column name(s) as `categorical_feature=`. No one-hot encoding needed.


In [5]:
X_with_cat = X_clf.copy()
X_with_cat['cat'] = pd.Categorical(rng.choice(['a', 'b', 'c'], size=len(X_with_cat)))
X_with_cat['cat'] = X_with_cat['cat'].cat.codes   # LightGBM wants integer codes

m = lgb.LGBMClassifier(n_estimators=100, verbosity=-1).fit(
    X_with_cat[:400], y_clf[:400], categorical_feature=['cat'],
)
print(f'val acc with cat: {m.score(X_with_cat[400:], y_clf[400:]):.3f}')

val acc with cat: 0.955


*Why LightGBM's categorical handling is special*: it splits on subsets of category values rather than one-hot ridges. For high-cardinality categoricals (zip codes, user IDs), it's far better than one-hot. *Common mistake*: passing string values directly — LightGBM wants integer codes.


### How do I get feature importance?

Both libraries expose `.feature_importances_` after fitting.


In [6]:
imp_xgb = pd.Series(m_xgb.feature_importances_, index=X_clf.columns).sort_values(ascending=False)
print('XGB importances:')
print(imp_xgb.round(4).head())

# LightGBM exposes 'gain' (total gain) and 'split' (n splits) importance via the booster.
gain = pd.Series(m_lgb.booster_.feature_importance(importance_type='gain'), index=X_clf.columns).sort_values(ascending=False)
print('\nLGBM gain importances:')
print(gain.round(1).head())

XGB importances:
f6    0.5473
f0    0.1757
f5    0.0779
f1    0.0675
f3    0.0431
dtype: float32

LGBM gain importances:
f6    4023.6
f0     572.1
f3     250.8
f1     136.4
f2      62.6
dtype: float64


*Gain vs split*: gain weights by how much loss each split reduced; split counts how many times each feature was used. Gain is what you usually want. *Common mistake*: trusting tree feature importance over permutation importance — tree importance reflects model-internal use, not generalisation.


---
## 3. Regression and quantile regression

Same API for regression. Quantile regression is a one-flag change that gives prediction intervals for free.


### How do I fit a regression model?

`XGBRegressor()` / `LGBMRegressor()`. Identical interface to the classifiers.


In [7]:
m_lgb_reg = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, verbosity=-1).fit(X_reg[:400], y_reg[:400])
print(f'val r2: {m_lgb_reg.score(X_reg[400:], y_reg[400:]):.3f}')

val r2: 0.958


/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


*Why LightGBM regressor*: same speed advantage, plus quantile and Tweedie objectives built in. *Common mistake*: using `objective='regression'` (squared error) when your target has heavy tails — try `'regression_l1'` (MAE) or quantile regression instead.


### How do I do quantile regression for prediction intervals?

`objective='quantile', alpha=0.1` for the 10th percentile; train one model per quantile.


In [8]:
def fit_q(alpha):
    return lgb.LGBMRegressor(objective='quantile', alpha=alpha,
                              n_estimators=200, learning_rate=0.05,
                              verbosity=-1, random_state=0).fit(X_reg[:400], y_reg[:400])

m10 = fit_q(0.10)
m50 = fit_q(0.50)
m90 = fit_q(0.90)

p10 = m10.predict(X_reg[400:])
p50 = m50.predict(X_reg[400:])
p90 = m90.predict(X_reg[400:])

coverage = ((y_reg[400:] >= p10) & (y_reg[400:] <= p90)).mean()
print(f'empirical coverage of [p10, p90]: {coverage:.3f}    (target 0.80)')

empirical coverage of [p10, p90]: 0.575    (target 0.80)


/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


*Why train three models*: each quantile is its own optimisation problem. *Common mistake*: assuming p10 ≤ p50 ≤ p90 always holds — quantile crossing happens; sort per-row in production.


---
## 4. Saving / loading models

Both libraries support pickling via joblib; both also have native binary formats.


### How do I save and load a model?

joblib for any sklearn-compatible wrapper; native `save_model` / `load_model` for the booster object.


In [9]:
import joblib, tempfile, os

# joblib (works for both libraries' sklearn wrapper).
path = os.path.join(tempfile.gettempdir(), 'lgbm_demo.joblib')
joblib.dump(m_lgb_reg, path)
m_loaded = joblib.load(path)
print('joblib roundtrip OK')

# LightGBM native format (smaller, version-safe).
booster_path = os.path.join(tempfile.gettempdir(), 'lgbm_demo.txt')
m_lgb_reg.booster_.save_model(booster_path)
print(f'native booster saved to {booster_path}')

joblib roundtrip OK
native booster saved to /tmp/lgbm_demo.txt


*When to use which*: joblib if you want to persist the sklearn-style wrapper (with the metadata it carries). Native format if you only need the trees and want maximum compatibility across library versions.


---
## 5. More: built-in CV, monotonic constraints, custom objectives

Three patterns to know once you're past 'just call .fit'.


### How do I run cross-validation with the library's built-in CV?

`lgb.cv(params, lgb_data, ..., folds=...)` and `xgb.cv(...)`. Faster than wrapping in sklearn for simple cases.


In [10]:
import lightgbm as lgb
from sklearn.datasets import make_regression
from sklearn.model_selection import TimeSeriesSplit

X, y = make_regression(n_samples=500, n_features=4, noise=5, random_state=0)
ds = lgb.Dataset(X, label=y)

# CV with a custom fold iterator (TimeSeriesSplit for sequential data).
result = lgb.cv(
    {'objective': 'regression_l1', 'learning_rate': 0.05, 'num_leaves': 31, 'verbosity': -1},
    ds, num_boost_round=200,
    folds=TimeSeriesSplit(3),
    callbacks=[lgb.early_stopping(20, verbose=False)],
)
# `result` is a dict mapping metric name to per-iteration mean across folds.
print(f'best l1 mean: {min(result["valid l1-mean"]):.4f}')

/home/zlac116/Code/learning/ml-revision/.venv/lib/python3.12/site-packages/sklearn/model_selection/_split.py:1262: UserWarning: The groups parameter is ignored by TimeSeriesSplit
  warnings.warn(


best l1 mean: 14.9928


*When to use*: lightweight CV without the sklearn wrapping overhead; native early-stopping support. *Common mistake*: forgetting `folds=TimeSeriesSplit(...)` and getting random K-fold by default (broken for time series).


### How do I impose monotonic constraints?

`monotone_constraints=` — for each feature, +1 (must increase prediction), -1 (must decrease), or 0 (no constraint).


In [11]:
import lightgbm as lgb
import numpy as np

# Make a simple regression where feature 0 should be monotonically increasing in y.
rng = np.random.default_rng(0)
X = rng.uniform(0, 1, (500, 3))
y = 2 * X[:, 0] + 0.1 * rng.normal(0, 1, 500)   # only f0 affects y, monotonically

# Without constraint — fit might be wiggly.
m = lgb.LGBMRegressor(n_estimators=200, verbosity=-1, random_state=0).fit(X, y)

# With constraint: feature 0 strictly increasing in prediction; others unconstrained.
mc = lgb.LGBMRegressor(
    n_estimators=200, verbosity=-1, random_state=0,
    monotone_constraints=[1, 0, 0],
).fit(X, y)

*When to use*: domain knowledge says feature X should always increase prediction (e.g. funding rate → vol). Constraint prevents weird non-monotonic behaviour from noise. *Common mistake*: imposing a constraint without testing it actually holds in your data — if reality is non-monotonic, you've crippled the model.


### How do I write a custom objective function?

Function returning `(grad, hess)` — first and second derivatives of your loss w.r.t. predictions.


In [12]:
import numpy as np
import lightgbm as lgb
from sklearn.datasets import make_regression

X, y = make_regression(n_samples=500, n_features=4, noise=5, random_state=0)

# Custom MAE objective: L = |y - ŷ|; grad = sign(ŷ - y); hess = constant ε.
def mae_objective(y_true, y_pred):
    grad = np.sign(y_pred - y_true)
    hess = np.full_like(y_pred, 0.01)   # smooth the second derivative
    return grad, hess

m = lgb.LGBMRegressor(
    n_estimators=200, learning_rate=0.05, verbosity=-1, random_state=0,
    objective=mae_objective,
).fit(X, y)

*When to use*: cost-asymmetric problems (over-prediction much worse than under-), domain-specific losses. *Common mistake*: returning hess=0 (mathematically correct for MAE) — LightGBM uses `grad/hess` to size splits, dividing by zero. Use a tiny constant.
